In [1]:
import arcpy
import os
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = None
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/Wetland Mapping"
nhd_gdb = os.path.normpath(os.path.join(project_folder, "Illinois_NHD_Shapefiles.gdb"))

In [2]:
# non-isolated flowline layer
step2_nhd = os.path.normpath(os.path.join(nhd_gdb, "NHDFlowline_Step2_IL_NonIsolated"))

In [25]:
# current NHD flowline layer
step3_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step3_IL_StreamClassification"))

In [4]:
# copy previous layer in case of mistakes
arcpy.conversion.ExportFeatures(
    in_features = step2_nhd,
    out_features = step3_nhd
);

# Perennial

## Unassigned flowlines that intersect the NHD area layer (streams, rivers, canals)

In [5]:
nhd_area_streams_rivers_canals = os.path.normpath(os.path.join(nhd_gdb,"NHDArea_StreamRiversCanals"))

In [6]:
update_perennial_rule = """
def UpdateHydroClass(NHD_Hydro_Class):
    if NHD_Hydro_Class == 0:
        return 1
    else:
        return NHD_Hydro_Class
"""

In [7]:
# starting perennial count
arcpy.management.MakeFeatureLayer(step2_nhd, "temp_layer");
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="temp_layer",
    selection_type="NEW_SELECTION",
    where_clause="NHD_Hydro_Class = 1"
);
count2 = int(arcpy.management.GetCount("temp_layer")[0])
arcpy.management.Delete("temp_layer");
count2

49304

In [8]:
arcpy.management.MakeFeatureLayer(step3_nhd, "temp_layer");
arcpy.management.SelectLayerByLocation(
    in_layer="temp_layer",
    overlap_type="INTERSECT",
    select_features=nhd_area_streams_rivers_canals,
    selection_type="NEW_SELECTION"
);
arcpy.management.CalculateField(
    in_table="temp_layer",
    field="NHD_Hydro_Class",
    expression = "UpdateHydroClass(!NHD_Hydro_Class!)",
    expression_type="PYTHON3",
    code_block = update_perennial_rule
);
arcpy.management.Delete("temp_layer");

In [9]:
arcpy.management.MakeFeatureLayer(step3_nhd, "temp_layer");
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="temp_layer",
    selection_type="NEW_SELECTION",
    where_clause="NHD_Hydro_Class = 1"
);
count3 = int(arcpy.management.GetCount("temp_layer")[0])
arcpy.management.Delete("temp_layer");

In [10]:
count3-count2

21654

## Unassigned flowlines that intersect perennial waterbodies

In [11]:
nhd_waterbody_perennial = os.path.normpath(os.path.join(nhd_gdb,"NHDWaterbody_IL_Perennial"))

In [12]:
arcpy.management.MakeFeatureLayer(step3_nhd, "temp_layer");
arcpy.management.SelectLayerByLocation(
    in_layer="temp_layer",
    overlap_type="INTERSECT",
    select_features=nhd_waterbody_perennial,
    selection_type="NEW_SELECTION"
);
arcpy.management.CalculateField(
    in_table="temp_layer",
    field="NHD_Hydro_Class",
    expression = "UpdateHydroClass(!NHD_Hydro_Class!)",
    expression_type="PYTHON3",
    code_block = update_perennial_rule
);
arcpy.management.Delete("temp_layer");

In [13]:
arcpy.management.MakeFeatureLayer(step3_nhd, "temp_layer");
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="temp_layer",
    selection_type="NEW_SELECTION",
    where_clause="NHD_Hydro_Class = 1"
);
count4 = int(arcpy.management.GetCount("temp_layer")[0])
arcpy.management.Delete("temp_layer");

In [14]:
count4-count3

32260

# Intermittment

## Unassigned flowlines that intersect intermittent waterbodies

In [15]:
nhd_waterbody_intermittent = os.path.normpath(os.path.join(nhd_gdb,"NHDWaterbody_IL_Intermittent"))

In [16]:
update_intermittent_rule = """
def UpdateHydroClass(NHD_Hydro_Class):
    if NHD_Hydro_Class == 0:
        return 2
    else:
        return NHD_Hydro_Class
"""

In [18]:
# starting intermittent count
arcpy.management.MakeFeatureLayer(step2_nhd, "temp_layer");
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="temp_layer",
    selection_type="NEW_SELECTION",
    where_clause="NHD_Hydro_Class = 2"
);
count5 = int(arcpy.management.GetCount("temp_layer")[0])
arcpy.management.Delete("temp_layer");

In [17]:
arcpy.management.MakeFeatureLayer(step3_nhd, "temp_layer");
arcpy.management.SelectLayerByLocation(
    in_layer="temp_layer",
    overlap_type="INTERSECT",
    select_features=nhd_waterbody_intermittent,
    selection_type="NEW_SELECTION"
);
arcpy.management.CalculateField(
    in_table="temp_layer",
    field="NHD_Hydro_Class",
    expression = "UpdateHydroClass(!NHD_Hydro_Class!)",
    expression_type="PYTHON3",
    code_block = update_intermittent_rule
);
arcpy.management.Delete("temp_layer");

In [19]:
arcpy.management.MakeFeatureLayer(step3_nhd, "temp_layer");
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="temp_layer",
    selection_type="NEW_SELECTION",
    where_clause="NHD_Hydro_Class = 2"
);
count6 = int(arcpy.management.GetCount("temp_layer")[0])
arcpy.management.Delete("temp_layer");

In [20]:
count6 - count5

1471

# Export attribute table to CSV

In [24]:
# save new file
step4_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step4_IL_WaterbodyIntersect_Classification"))

In [26]:
# copy previous layer in case of mistakes
arcpy.conversion.ExportFeatures(
    in_features = step3_nhd,
    out_features = step4_nhd
);

In [21]:
output_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/JEMA-Paper-Results/NHD_Data"
out_table = os.path.normpath(os.path.join(output_folder, "NHDFlowline_IL_Step4_HydroClass.csv"))

In [22]:
arcpy.conversion.ExportTable(step4_nhd, out_table);

<Result 'C:\\Users\\Chels\\OneDrive - University of Illinois - Urbana\\Ch5_Sackett_Impacts\\JEMA-Paper-Results\\NHD_Data\\NHDFlowline_IL_Step4_HydroClass.csv'>